# 01 — Significance tests

**Question:** Is the conversion rate significantly higher during Black Friday–Cyber Monday than on normal days?

**Data:** BigQuery query `sql/08_black_friday_vs_normal.sql` on the GA4 obfuscated sample (Google Merchandise Store). Unit: distinct users who viewed at least one product. Normal days: Nov 1–20, 2020 (14,041 viewers, 866 buyers). Black Friday–Cyber Monday: Nov 27–30, 2020 (3,225 viewers, 333 buyers).

**Method:** Two-sided two-proportion z-test with a pooled rate, α = 5% (fixed before looking at the result). Computed by hand with NumPy, then checked with `statsmodels`. We also report the 95% confidence interval of the difference.

In [3]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
print("ok")

ok


In [7]:
# Data (query 08)
x_normal, n_normal = 866, 14041   # buyers, viewers
x_bf, n_bf = 333, 3225

# Step 2: conversion rate of each group
p_normal = x_normal / n_normal
p_bf = x_bf / n_bf

# Step 3: pooled rate under H0
p_pool = (x_normal + x_bf) / (n_normal + n_bf)

# Step 4: standard error under H0, then z
se = np.sqrt(p_pool * (1-p_pool) * (1 / n_normal + 1 / n_bf))
z = (p_bf - p_normal) / se

# Step 5: two-sided p-value
p_value = 2 * stats.norm.sf(abs(z))

print(f"p_normal = {p_normal:.4f}, p_bf = {p_bf:.4f}")
print(f"z = {z:.2f}, p-value = {p_value:.2e}")

p_normal = 0.0617, p_bf = 0.1033
z = 8.38, p-value = 5.45e-17


In [9]:
from statsmodels.stats.proportion import confint_proportions_2indep

# Check with statsmodels (BF first so that z > 0)
z_sm, p_sm = proportions_ztest(count=[333, 866], nobs=[3225, 14041])
print(f"statsmodels: z = {z_sm:.2f}, p-value = {p_sm:.2e}")

# 95% CI of the difference p_bf - p_normal
low, high = confint_proportions_2indep(333, 3225, 866, 14041, method="wald")
print(f"Difference = {100*(p_bf - p_normal):.1f} pts, 95% CI = [{100*low:.1f} ; {100*high:.1f}] pts")
print(f"Relative lift = {100*(p_bf/p_normal - 1):.0f}%")

statsmodels: z = 8.38, p-value = 5.45e-17
Difference = 4.2 pts, 95% CI = [3.0 ; 5.3] pts
Relative lift = 67%


## Conclusion

Conversion is significantly higher during Black Friday–Cyber Monday than on normal days: 10.3% vs 6.2% (z = 8.38, p < 0.001), so the gap is very unlikely to be due to chance.

The effect is large: +4.2 percentage points (95% CI: +3.0 to +5.3 pts), i.e. a +67% relative lift.

**Limits:** this is an observational comparison, not a randomized experiment, so it shows *that* conversion was higher, not *why* (discounts, ads, holiday shoppers already decided to buy). The two periods also have different lengths (20 vs 4 days), and the same user can appear in both groups, which breaks the independence assumption of the test.